# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [2]:

from pathlib import Path

import duckdb
import polars as pl

# Lokasi project
PROJECT_ROOT = Path.cwd().resolve().parent

# Lokasi dataset CSV dan output Parquet
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "SFinDSet.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

# Pastikan dataset tersedia
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Pastikan SFinDSet.csv berada di data/raw/."
    )

# Baca dataset CSV
df = pl.read_csv(DATA_PATH)

print(f"Dataset: {DATA_PATH.name}")
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")

# Tampilkan 5 baris pertama
df.head()


Dataset: SFinDSet.csv
Ukuran awal: 1,048,575 baris x 9 kolom


Transaction_ID,Account_Number,Account_Name,Transaction_Amount,Transaction_Mode,Transaction_Type,Transaction_Date,Transaction_Source,Transaction_Destination
str,i64,str,f64,str,str,str,i64,i64
"""TRANS-588469""",9368326206,"""David Howard""",1834602.8,"""Card Out""","""NGN to USD""","""10/28/2021 2:31""",1698161104,20322866
"""TRANS-239018""",6826045758,"""David Hughes""",537724.4,"""Card Out""","""NGN to USD""","""8/31/2022 23:38""",8172779956,20322866
"""TRANS-1949426""",5386299916,"""David Huber""",7.875963e6,"""Card Out""","""NGN to USD""","""8/20/2023 8:56""",3138972375,20322866
"""TRANS-1588193""",1443563124,"""David Hunt""",2.0079e6,"""Card Out""","""NGN to USD""","""2/13/2023 16:14""",9694080118,20322866
"""TRANS-224390""",7244060183,"""David Hughes""",4.7660e7,"""Card Out""","""NGN to USD""","""3/21/2021 17:10""",6225683619,20322866


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [4]:

connection = duckdb.connect()
connection.register("dataset", df)

profil = pl.from_pandas(
    connection.execute(
        "SUMMARIZE SELECT * FROM dataset"
    ).fetchdf()
)

profil


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,f64
"""Transaction_ID""","""VARCHAR""","""TRANS-1""","""TRANS-999999""",1013351,null,null,null,null,null,1048575,0.0
"""Account_Number""","""BIGINT""","""500""","""9999995264""",1020194,"""4998590299.811408""","""2886773064.090957""","""2500861016""","""5000891975""","""7500210615""",1048575,0.0
"""Account_Name""","""VARCHAR""","""Aaron Abbott""","""Zoe Zavala""",333013,null,null,null,null,null,1048575,0.0
"""Transaction_Amount""","""DOUBLE""","""100.5""","""49999946.5""",1100482,"""13764588.539523648""","""15221204.816237042""","""2279999.7357983734""","""4548594.68001335""","""25065380.88096227""",1048575,0.0
"""Transaction_Mode""","""VARCHAR""","""Card Out""","""Transfer Out""",5,null,null,null,null,null,1048575,0.0
"""Transaction_Type""","""VARCHAR""","""NGN to NGN""","""USD to USD""",4,null,null,null,null,null,1048575,0.0
"""Transaction_Date""","""VARCHAR""","""1/1/2020 0:01""","""9/9/2023 9:58""",813717,null,null,null,null,null,1048575,0.0
"""Transaction_Source""","""BIGINT""","""12599""","""9999999517""",1041513,"""4998512684.282938""","""2886117872.4148717""","""2495494525""","""4998009160""","""7495818620""",1048575,0.0
"""Transaction_Destination""","""BIGINT""","""4959""","""9999997549""",1005822,"""4998704461.818359""","""2885133086.3944592""","""2503754025""","""4997443528""","""7495619724""",1048575,0.0


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [5]:

jumlah_baris = len(df)

if jumlah_baris == 0:
    raise ValueError("Dataset kosong, tidak bisa menghitung persentase null.")

null_summary = (
    df.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .with_columns(
        (
            pl.col("jumlah_null") / jumlah_baris * 100
        ).round(2).alias("persen_null")
    )
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)

null_summary


kolom,jumlah_null,persen_null
str,u32,f64


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [6]:

# Tangani nilai null secara otomatis berdasarkan tipe data kolom

jumlah_null_sebelum = df.null_count().sum_horizontal().item()

numerik = [
    kolom for kolom, tipe in df.schema.items()
    if tipe.is_numeric()
]

kategorik = [
    kolom for kolom, tipe in df.schema.items()
    if tipe == pl.String
]

ekspresi = []

# Kolom numerik: isi null dengan median
for kolom in numerik:
    median = df[kolom].median()

    if median is not None:
        ekspresi.append(
            pl.col(kolom).fill_null(median)
        )

# Kolom teks/kategorik: isi null dengan "tidak diketahui"
for kolom in kategorik:
    ekspresi.append(
        pl.col(kolom).fill_null("tidak diketahui")
    )

# Terapkan penanganan null
df_missing_handled = (
    df.with_columns(ekspresi)
    if ekspresi
    else df.clone()
)

jumlah_null_sesudah = (
    df_missing_handled.null_count()
    .sum_horizontal()
    .item()
)

print(f"Null sebelum : {jumlah_null_sebelum:,}")
print(f"Null sesudah : {jumlah_null_sesudah:,}")

print("\nUkuran dataset:", df_missing_handled.shape)
print("\nSisa null per kolom:")
print(
    df_missing_handled.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .filter(pl.col("jumlah_null") > 0)
)

df_missing_handled.head()


Null sebelum : 0
Null sesudah : 0

Ukuran dataset: (1048575, 9)

Sisa null per kolom:
shape: (0, 2)
┌───────┬─────────────┐
│ kolom ┆ jumlah_null │
│ ---   ┆ ---         │
│ str   ┆ u32         │
╞═══════╪═════════════╡
└───────┴─────────────┘


Transaction_ID,Account_Number,Account_Name,Transaction_Amount,Transaction_Mode,Transaction_Type,Transaction_Date,Transaction_Source,Transaction_Destination
str,f64,str,f64,str,str,str,f64,f64
"""TRANS-588469""",9.3683e9,"""David Howard""",1834602.8,"""Card Out""","""NGN to USD""","""10/28/2021 2:31""",1.6982e9,2.0322866e7
"""TRANS-239018""",6.8260e9,"""David Hughes""",537724.4,"""Card Out""","""NGN to USD""","""8/31/2022 23:38""",8.1728e9,2.0322866e7
"""TRANS-1949426""",5.3863e9,"""David Huber""",7.875963e6,"""Card Out""","""NGN to USD""","""8/20/2023 8:56""",3.1390e9,2.0322866e7
"""TRANS-1588193""",1.4436e9,"""David Hunt""",2.0079e6,"""Card Out""","""NGN to USD""","""2/13/2023 16:14""",9.6941e9,2.0322866e7
"""TRANS-224390""",7.2441e9,"""David Hughes""",4.7660e7,"""Card Out""","""NGN to USD""","""3/21/2021 17:10""",6.2257e9,2.0322866e7


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [12]:

import polars as pl

# Gunakan DataFrame hasil penanganan missing values
KOLOM_NUMERIK = next(
    (
        kolom
        for kolom, tipe in df_missing_handled.schema.items()
        if tipe.is_numeric()
    ),
    None
)

if KOLOM_NUMERIK is None:
    raise ValueError("Tidak ditemukan kolom numerik.")

data = df_missing_handled[KOLOM_NUMERIK].drop_nulls()

q1 = data.quantile(0.25)
q3 = data.quantile(0.75)
iqr = q3 - q1

batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print("Kolom:", KOLOM_NUMERIK)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Batas bawah:", batas_bawah)
print("Batas atas:", batas_atas)

df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(
        lower_bound=batas_bawah,
        upper_bound=batas_atas
    )
)

print("Jumlah baris:", df_outlier_handled.height)
df_outlier_handled.head()


Kolom: Account_Number
Q1: 2498671656.0
Q3: 7498338387.0
IQR: 4999666731.0
Batas bawah: -5000828440.5
Batas atas: 14997838483.5
Jumlah baris: 1048575


Transaction_ID,Account_Number,Account_Name,Transaction_Amount,Transaction_Mode,Transaction_Type,Transaction_Date,Transaction_Source,Transaction_Destination
str,f64,str,f64,str,str,str,f64,f64
"""TRANS-588469""",9.3683e9,"""David Howard""",1834602.8,"""Card Out""","""NGN to USD""","""10/28/2021 2:31""",1.6982e9,2.0322866e7
"""TRANS-239018""",6.8260e9,"""David Hughes""",537724.4,"""Card Out""","""NGN to USD""","""8/31/2022 23:38""",8.1728e9,2.0322866e7
"""TRANS-1949426""",5.3863e9,"""David Huber""",7.875963e6,"""Card Out""","""NGN to USD""","""8/20/2023 8:56""",3.1390e9,2.0322866e7
"""TRANS-1588193""",1.4436e9,"""David Hunt""",2.0079e6,"""Card Out""","""NGN to USD""","""2/13/2023 16:14""",9.6941e9,2.0322866e7
"""TRANS-224390""",7.2441e9,"""David Hughes""",4.7660e7,"""Card Out""","""NGN to USD""","""3/21/2021 17:10""",6.2257e9,2.0322866e7


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [13]:

# Periksa duplikat pada dataset
total_baris = len(df_outlier_handled)

# Cari kolom yang berpotensi menjadi ID unik
kandidat_id = [
    kolom for kolom in df_outlier_handled.columns
    if kolom.lower() in ["id", "id_unik", "record_id", "transaction_id"]
]

if kandidat_id:
    KOLOM_KUNCI = kandidat_id[0]

    baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique()

    print(f"Kolom kunci: {KOLOM_KUNCI}")
    print(f"Total baris: {total_baris:,}")
    print(f"Nilai unik pada kolom kunci: {baris_unik:,}")

    # Hapus baris dengan kunci yang sama, pertahankan kemunculan pertama
    df_dedup = df_outlier_handled.unique(
        subset=[KOLOM_KUNCI],
        keep="first",
        maintain_order=True
    )
else:
    # Jika tidak ada kolom ID, cek duplikat seluruh baris
    df_dedup = df_outlier_handled.unique(
        subset=None,
        keep="first",
        maintain_order=True
    )

    print("Kolom ID khusus tidak ditemukan.")
    print("Pemeriksaan dilakukan berdasarkan seluruh kolom.")

baris_setelah = len(df_dedup)

print(f"Baris setelah deduplikasi: {baris_setelah:,}")
print(f"Duplikat yang dihapus: {total_baris - baris_setelah:,}")

df_dedup.head()


Kolom kunci: Transaction_ID
Total baris: 1,048,575
Nilai unik pada kolom kunci: 1,044,504
Baris setelah deduplikasi: 1,044,504
Duplikat yang dihapus: 4,071


Transaction_ID,Account_Number,Account_Name,Transaction_Amount,Transaction_Mode,Transaction_Type,Transaction_Date,Transaction_Source,Transaction_Destination
str,f64,str,f64,str,str,str,f64,f64
"""TRANS-588469""",9.3683e9,"""David Howard""",1834602.8,"""Card Out""","""NGN to USD""","""10/28/2021 2:31""",1.6982e9,2.0322866e7
"""TRANS-239018""",6.8260e9,"""David Hughes""",537724.4,"""Card Out""","""NGN to USD""","""8/31/2022 23:38""",8.1728e9,2.0322866e7
"""TRANS-1949426""",5.3863e9,"""David Huber""",7.875963e6,"""Card Out""","""NGN to USD""","""8/20/2023 8:56""",3.1390e9,2.0322866e7
"""TRANS-1588193""",1.4436e9,"""David Hunt""",2.0079e6,"""Card Out""","""NGN to USD""","""2/13/2023 16:14""",9.6941e9,2.0322866e7
"""TRANS-224390""",7.2441e9,"""David Hughes""",4.7660e7,"""Card Out""","""NGN to USD""","""3/21/2021 17:10""",6.2257e9,2.0322866e7


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [14]:

# Daftarkan DataFrame hasil deduplikasi ke DuckDB
connection.register("dataset_bersih", df_dedup)

# Validasi data setelah pembersihan
# Gunakan nama kolom numerik yang sudah dipilih sebelumnya
kolom_sql = '"' + KOLOM_NUMERIK.replace('"', '""') + '"'

query_validasi = f"""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) FILTER (
            WHERE {kolom_sql} IS NULL
        ) AS pelanggaran_null,
        COUNT(*) FILTER (
            WHERE {kolom_sql} < ?
               OR {kolom_sql} > ?
        ) AS pelanggaran_batas
    FROM dataset_bersih
"""

validasi = pl.from_pandas(
    connection.execute(
        query_validasi,
        [batas_bawah, batas_atas]
    ).fetchdf()
)

validasi


total_baris,pelanggaran_null,pelanggaran_batas
i64,i64,i64
1044504,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [15]:

from pathlib import Path

# Lokasi penyimpanan hasil cleaning
OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
)

# Buat folder processed jika belum tersedia
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Simpan dataset yang sudah dibersihkan
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning : {len(df):,}")
print(f"Baris sesudah cleaning : {len(df_dedup):,}")
print(f"Baris yang dihapus     : {len(df) - len(df_dedup):,}")
print(f"Kolom dataset          : {df_dedup.shape[1]}")
print(f"Ukuran file output     : {OUTPUT_PATH.stat().st_size / 1_000_000:.2f} MB")
print(f"Data bersih tersimpan  : {OUTPUT_PATH}")


Baris sebelum cleaning : 1,048,575
Baris sesudah cleaning : 1,044,504
Baris yang dihapus     : 4,071
Kolom dataset          : 9
Ukuran file output     : 34.78 MB
Data bersih tersimpan  : /home/jovyan/work/data/processed/dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

In [16]:

# ==========================================
# DOKUMENTASI HASIL DATA CLEANING
# ==========================================

print("=" * 55)
print("DOKUMENTASI DATA CLEANING - SFinDSet")
print("=" * 55)

# 1. Dimensi kualitas data yang paling bermasalah
null_per_kolom = df.null_count().row(0, named=True)
total_null = sum(null_per_kolom.values())

kolom_null_terbanyak = (
    max(null_per_kolom, key=null_per_kolom.get)
    if null_per_kolom else None
)

duplikat_dihapus = len(df_outlier_handled) - len(df_dedup)
baris_berubah = len(df) - len(df_dedup)

print("\n1. DIMENSI KUALITAS DATA")
print(f"Total nilai null sebelum cleaning: {total_null:,}")

if kolom_null_terbanyak and total_null > 0:
    print(f"Kolom dengan null terbanyak: {kolom_null_terbanyak}")
    print(f"Jumlah null: {null_per_kolom[kolom_null_terbanyak]:,}")
else:
    print("Tidak ditemukan nilai null pada dataset awal.")

print(f"Jumlah duplikat yang dihapus: {duplikat_dihapus:,}")

# 2. Strategi missing values
null_awal = df.null_count().sum_horizontal().item()
null_akhir = df_missing_handled.null_count().sum_horizontal().item()

print("\n2. STRATEGI MISSING VALUES")
print("Strategi: median untuk kolom numerik dan")
print("'tidak diketahui' untuk kolom teks.")
print(f"Null sebelum penanganan: {null_awal:,}")
print(f"Null sesudah penanganan: {null_akhir:,}")

# 3. Strategi outlier
data_outlier_awal = df_missing_handled[KOLOM_NUMERIK]
outlier_awal = (
    (data_outlier_awal < batas_bawah)
    | (data_outlier_awal > batas_atas)
).sum()

data_outlier_akhir = df_outlier_handled[KOLOM_NUMERIK]
outlier_akhir = (
    (data_outlier_akhir < batas_bawah)
    | (data_outlier_akhir > batas_atas)
).sum()

print("\n3. STRATEGI OUTLIER")
print(f"Kolom yang diperiksa: {KOLOM_NUMERIK}")
print("Metode: IQR (Interquartile Range)")
print("Penanganan: Winsorization menggunakan clipping")
print(f"Batas bawah: {batas_bawah}")
print(f"Batas atas: {batas_atas}")
print(f"Outlier sebelum: {outlier_awal:,}")
print(f"Outlier sesudah: {outlier_akhir:,}")

# 4. Perubahan jumlah baris
print("\n4. PERUBAHAN JUMLAH BARIS")
print(f"Baris awal: {len(df):,}")
print(f"Baris setelah missing values: {len(df_missing_handled):,}")
print(f"Baris setelah outlier: {len(df_outlier_handled):,}")
print(f"Baris setelah deduplikasi: {len(df_dedup):,}")
print(f"Perubahan bersih jumlah baris: {baris_berubah:,}")

# 5. Aturan validasi
print("\n5. ATURAN VALIDASI")
print("- Memeriksa jumlah baris dataset.")
print("- Memeriksa nilai null pada kolom numerik.")
print("- Memastikan nilai kolom numerik berada dalam batas IQR.")

print("\nHasil validasi:")
print(validasi)

# 6. Lokasi output
print("\n6. OUTPUT DATA CLEANING")
print(f"Lokasi file: {OUTPUT_PATH}")
print(f"File tersedia: {OUTPUT_PATH.exists()}")
print(f"Ukuran file: {OUTPUT_PATH.stat().st_size / 1_000_000:.2f} MB"
      if OUTPUT_PATH.exists() else "File output belum tersedia.")

print("\nDokumentasi selesai.")


DOKUMENTASI DATA CLEANING - SFinDSet

1. DIMENSI KUALITAS DATA
Total nilai null sebelum cleaning: 0
Tidak ditemukan nilai null pada dataset awal.
Jumlah duplikat yang dihapus: 4,071

2. STRATEGI MISSING VALUES
Strategi: median untuk kolom numerik dan
'tidak diketahui' untuk kolom teks.
Null sebelum penanganan: 0
Null sesudah penanganan: 0

3. STRATEGI OUTLIER
Kolom yang diperiksa: Account_Number
Metode: IQR (Interquartile Range)
Penanganan: Winsorization menggunakan clipping
Batas bawah: -5000828440.5
Batas atas: 14997838483.5
Outlier sebelum: 0
Outlier sesudah: 0

4. PERUBAHAN JUMLAH BARIS
Baris awal: 1,048,575
Baris setelah missing values: 1,048,575
Baris setelah outlier: 1,048,575
Baris setelah deduplikasi: 1,044,504
Perubahan bersih jumlah baris: 4,071

5. ATURAN VALIDASI
- Memeriksa jumlah baris dataset.
- Memeriksa nilai null pada kolom numerik.
- Memastikan nilai kolom numerik berada dalam batas IQR.

Hasil validasi:
shape: (1, 3)
┌─────────────┬──────────────────┬──────────────